In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml
dbutils.library.restartPython()


In [0]:
# =============================================================================
# Carrega a função compartilhada que atualiza o status real da fonte na
# tabela de controle (desafio_kinea.research.controle_fontes) a cada execução.
# =============================================================================
%run "/Workspace/Shared/Research_Infra/Data-Ingestion-Pipeline-for-Kinea-Research-Infrastructure/scripts/utils_controle"


In [0]:
import os
import re
import json
import time
import random
import hashlib
import unicodedata
import urllib.parse
from datetime import datetime, timezone
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests


In [0]:
try:
    dbutils.widgets.remove("fonte")
except Exception:
    pass

dbutils.widgets.text("fonte", "todas")
NOME_FONTE = dbutils.widgets.get("fonte")

HOJE = datetime.now(timezone.utc).astimezone().strftime("%Y-%m-%d")

PASTA_DESTINO = f"/Volumes/desafio_kinea/research/research_volume/infraestrutura/files/{HOJE}"
os.makedirs(PASTA_DESTINO, exist_ok=True)

PASTA_MANIFESTOS = "/Volumes/desafio_kinea/research/research_volume/infraestrutura/manifests"
os.makedirs(PASTA_MANIFESTOS, exist_ok=True)

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
]

IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]
HTTP_TIMEOUT = 30
MIN_CHARS_TEXTO = 200

TAGS_LIXO = ["script", "style", "noscript", "iframe", "svg", "form",
             "nav", "header", "footer", "aside", "button"]
SELETORES_CONTEUDO = ["#content-core", "#parent-fieldname-text", "#content", "main"]


In [0]:
def slugify(texto: str, max_len: int = 80) -> str:
    if not texto:
        return "sem-titulo"
    nfkd = unicodedata.normalize("NFKD", texto)
    ascii_txt = nfkd.encode("ascii", "ignore").decode("ascii")
    ascii_txt = re.sub(r"[^a-zA-Z0-9]+", "-", ascii_txt).strip("-").lower()
    return (ascii_txt[:max_len] or "sem-titulo").strip("-")


def hash_curto(texto: str, n: int = 8) -> str:
    return hashlib.md5(texto.encode("utf-8")).hexdigest()[:n]


def headers_aleatorios(referer: Optional[str] = None) -> dict:
    headers = {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Accept-Encoding": "gzip, deflate",
    }
    if referer:
        headers["Referer"] = referer
    return headers


def carregar_manifesto(caminho: str) -> set:
    if not os.path.exists(caminho):
        return set()
    try:
        with open(caminho, "r", encoding="utf-8") as f:
            return set(json.load(f))
    except Exception:
        return set()


def salvar_manifesto(caminho: str, urls: set) -> None:
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(sorted(urls), f, ensure_ascii=False, indent=2)


def baixar_pagina(url: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer=url)
        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None


def extrair_texto_generico(html: str) -> str:
    try:
        soup = BeautifulSoup(html, "lxml")
    except Exception:
        soup = BeautifulSoup(html, "html.parser")

    for tag in soup(TAGS_LIXO):
        tag.decompose()

    base = None
    for seletor in SELETORES_CONTEUDO:
        encontrado = soup.select_one(seletor)
        if encontrado and len(encontrado.get_text(strip=True)) > 300:
            base = encontrado
            break
    if base is None:
        article = soup.find("article")
        base = article if (article and len(article.get_text(strip=True)) > 500) else soup

    texto = base.get_text("\n", strip=True)
    return re.sub(r"\n{3,}", "\n\n", texto).strip()


def extrair_titulo_h1(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    h1 = soup.find("h1")
    if h1:
        texto = h1.get_text(" ", strip=True)
        return texto or None
    return None


In [0]:
# --- Acende Brasil ---

PADRAO_URL_ACENDE = re.compile(r"/artigo/[a-z0-9\-]+/?$", re.IGNORECASE)
PADRAO_DATA_ACENDE = re.compile(r"Data da publica[çc][ãa]o:\s*(\d{2}/\d{2}/\d{4})")


def listar_acende_brasil(html: str, url_base: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    itens, vistos = [], set()
    for tag_a in soup.find_all("a", href=True):
        url_absoluta = urllib.parse.urljoin(url_base, tag_a["href"].strip())
        if not PADRAO_URL_ACENDE.search(url_absoluta) or url_absoluta in vistos:
            continue
        vistos.add(url_absoluta)
        itens.append({"titulo": tag_a.get_text(" ", strip=True), "url": url_absoluta})
    return itens


def data_acende_brasil(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    m = PADRAO_DATA_ACENDE.search(soup.get_text(" ", strip=True))
    if not m:
        return None
    dia, mes, ano = m.group(1).split("/")
    return f"{ano}-{mes}-{dia}"


# --- ANTT ---

COLECAO_BASE_ANTT = "https://www.gov.br/antt/pt-br/assuntos/noticias-defeso-eleitoral"
PADRAO_DATA_ANTT = re.compile(r"Publicado em\s*(\d{2}/\d{2}/\d{4})")


def listar_antt(html: str, url_base: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    itens, vistos = [], set()
    for tag_a in soup.find_all("a", href=True):
        url_absoluta = urllib.parse.urljoin(url_base, tag_a["href"].strip())
        if not url_absoluta.startswith(COLECAO_BASE_ANTT + "/") or "?" in url_absoluta:
            continue
        if url_absoluta in vistos:
            continue
        vistos.add(url_absoluta)
        titulo = tag_a.get_text(" ", strip=True)
        if titulo:
            itens.append({"titulo": titulo, "url": url_absoluta})
    return itens


def data_antt(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    m = PADRAO_DATA_ANTT.search(soup.get_text(" ", strip=True))
    if not m:
        return None
    dia, mes, ano = m.group(1).split("/")
    return f"{ano}-{mes}-{dia}"


# --- Agesan-RS Notícias ---

PADRAO_DATA_AGESAN = re.compile(r"(\d{1,2}/\d{2}/\d{2})\s")
TITULOS_EXCLUIR_AGESAN = ["extrato de aviso prévio"]


def listar_agesan_noticias(html: str, url_base: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    itens, vistos = [], set()
    for h2 in soup.find_all("h2"):
        tag_a = h2.find("a", href=True)
        if not tag_a:
            continue
        titulo = tag_a.get_text(" ", strip=True)
        if not titulo or any(t in titulo.lower() for t in TITULOS_EXCLUIR_AGESAN):
            continue
        url_absoluta = urllib.parse.urljoin(url_base, tag_a["href"].strip())
        if url_absoluta in vistos:
            continue
        vistos.add(url_absoluta)

        data_publicacao = None
        proximo = h2.find_next(string=PADRAO_DATA_AGESAN)
        if proximo:
            m = PADRAO_DATA_AGESAN.search(proximo)
            if m:
                dia, mes, ano = m.group(1).split("/")
                ano_completo = f"20{ano}" if len(ano) == 2 else ano
                data_publicacao = f"{ano_completo}-{mes}-{dia.zfill(2)}"

        itens.append({"titulo": titulo, "url": url_absoluta, "published_at": data_publicacao})
    return itens


# --- PSR (via Exame) ---

MESES_PT = {
    "janeiro": "01", "fevereiro": "02", "março": "03", "abril": "04",
    "maio": "05", "junho": "06", "julho": "07", "agosto": "08",
    "setembro": "09", "outubro": "10", "novembro": "11", "dezembro": "12",
}
PADRAO_DATA_EXAME = re.compile(r"(\d{1,2}) de (\w+) de (\d{4})")

CAMINHOS_EXCLUIR_EXAME = [
    "/politica-de-privacidade/", "/termos-de-uso/", "/politica-de-cookies/",
    "/lei-transparencia-salarial/", "/institucional/", "/newsletters/",
    "/canais-especiais/", "/faculdade", "/revista-exame/",
    "/invest/calculadoras/", "/invest/guia/",
    "materias-em-destaque",
]


def listar_psr_exame(html: str, url_base: str, max_paginas: int = 2) -> list[dict]:
    itens, vistos = [], set()

    for pagina in range(1, max_paginas + 1):
        url_pagina = url_base if pagina == 1 else f"{url_base.rstrip('/')}/{pagina}/"
        html_pagina = html if pagina == 1 else baixar_pagina(url_pagina)
        if not html_pagina:
            break

        soup = BeautifulSoup(html_pagina, "lxml")
        for tag_a in soup.find_all("a", href=True):
            href = tag_a["href"].strip()
            if "/arquivo/" in href or href in vistos:
                continue
            if not href.startswith("https://exame.com/") or href.rstrip("/") == "https://exame.com":
                continue
            if any(caminho in href for caminho in CAMINHOS_EXCLUIR_EXAME):
                continue
            titulo = tag_a.get_text(" ", strip=True)
            if len(titulo) < 20:
                continue
            vistos.add(href)
            itens.append({"titulo": titulo, "url": href})

    return itens


def data_psr_exame(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    m = PADRAO_DATA_EXAME.search(soup.get_text(" ", strip=True))
    if not m:
        return None
    dia, mes_nome, ano = m.groups()
    mes = MESES_PT.get(mes_nome.lower())
    if not mes:
        return None
    return f"{ano}-{mes}-{dia.zfill(2)}"


def extrair_titulo_h1_exame(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    for h1 in soup.find_all("h1"):
        texto = h1.get_text(" ", strip=True)
        if texto and len(texto) > 15 and texto.lower() not in ("esg", "economia", "negócios"):
            return texto
    return None


# --- ANEEL (mesma plataforma Plone do ANTT, generalizada) ---

PADRAO_DATA_GOVBR = re.compile(r"Publicado em\s*(\d{2}/\d{2}/\d{4})")


def listar_govbr_plone(html: str, url_base: str, colecao_base: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    itens, vistos = [], set()
    for tag_a in soup.find_all("a", href=True):
        url_absoluta = urllib.parse.urljoin(url_base, tag_a["href"].strip())
        if not url_absoluta.startswith(colecao_base + "/") or "?" in url_absoluta:
            continue
        if url_absoluta in vistos:
            continue
        vistos.add(url_absoluta)
        titulo = tag_a.get_text(" ", strip=True)
        if titulo:
            itens.append({"titulo": titulo, "url": url_absoluta})
    return itens


def data_govbr_plone(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    m = PADRAO_DATA_GOVBR.search(soup.get_text(" ", strip=True))
    if not m:
        return None
    dia, mes, ano = m.group(1).split("/")
    return f"{ano}-{mes}-{dia}"

# --- Agetransp ---

PADRAO_DATA_LISTAGEM_AGETRANSP = re.compile(r"(\d{2}/\d{2}/\d{4})")


def listar_agetransp(html: str, url_base: str) -> list[dict]:
    itens, vistos = [], set()

    for tag_a in BeautifulSoup(html, "lxml").find_all("a", href=True):
        href = tag_a["href"].strip()
        url_absoluta = urllib.parse.urljoin(url_base, href)

        if not url_absoluta.startswith("https://www.agetransp.rj.gov.br/noticias/"):
            continue
        if url_absoluta in vistos:
            continue

        titulo = tag_a.get_text(" ", strip=True)
        if len(titulo) < 15:
            continue
        vistos.add(url_absoluta)

        data_publicacao = None
        proximo_texto = tag_a.find_next(string=PADRAO_DATA_LISTAGEM_AGETRANSP)
        if proximo_texto:
            m = PADRAO_DATA_LISTAGEM_AGETRANSP.search(proximo_texto)
            if m:
                dia, mes, ano = m.group(1).split("/")
                data_publicacao = f"{ano}-{mes}-{dia}"

        itens.append({"titulo": titulo, "url": url_absoluta, "published_at": data_publicacao})

    return itens

In [0]:
CONFIGS_FONTES = {
    "acende_brasil": {
        "site_url": "https://acendebrasil.com.br/artigos/",
        "source_id": "acende_brasil",
        "source_descricao": "Linked from Instituto Acende Brasil — Artigos",
        "listar": listar_acende_brasil,
        "extrair_data": data_acende_brasil,
        "extrair_titulo": extrair_titulo_h1,
    },
    "antt": {
        "site_url": (
            "https://www.gov.br/antt/pt-br/assuntos/noticias-defeso-eleitoral"
            "?form.submitted=1&texto=&dt_inicio=&dt_fim=&categoria=infraestrutura-transito-e-transportes"
        ),
        "source_id": "antt_noticias_infraestrutura",
        "source_descricao": "Linked from ANTT — Notícias (Infraestrutura, Trânsito e Transportes)",
        "listar": listar_antt,
        "extrair_data": data_antt,
        "extrair_titulo": extrair_titulo_h1,
    },
    "agesan_noticias": {
        "site_url": "https://agesan-rs.com.br/noticias/",
        "source_id": "agesan_rs_noticias",
        "source_descricao": "Linked from Agesan-RS — Notícias",
        "listar": listar_agesan_noticias,
        "extrair_data": None,
        "extrair_titulo": None,
    },
    "psr_exame": {
        "site_url": "https://exame.com/arquivo/psr-energia-em-foco/",
        "source_id": "psr_energia_em_foco",
        "source_descricao": "Linked from Exame — PSR Energia em Foco",
        "listar": listar_psr_exame,
        "extrair_data": data_psr_exame,
        "extrair_titulo": extrair_titulo_h1_exame,
        "requer_data": True,
    },
    "aneel": {
        "site_url": "https://www.gov.br/aneel/pt-br/assuntos/noticias",
        "source_id": "aneel_noticias",
        "source_descricao": "Linked from ANEEL — Notícias",
        "listar": lambda html, url: listar_govbr_plone(
            html, url, "https://www.gov.br/aneel/pt-br/assuntos/noticias/2026-defeso-eleitoral"
        ),
        "extrair_data": data_govbr_plone,
        "extrair_titulo": extrair_titulo_h1,
    },
    "agetransp": {
        "site_url": "https://www.agetransp.rj.gov.br/noticias",
        "source_id": "agetransp_noticias",
        "source_descricao": "Linked from AGETRANSP — Notícias",
        "listar": listar_agetransp,
        "extrair_data": None,
        "extrair_titulo": None,
    },
}

In [0]:
def salvar_artefatos(pasta: str, source_id: str, titulo: str, texto: str, metadados: dict) -> tuple[str, str]:
    slug_source = slugify(source_id, max_len=40)
    slug_titulo = slugify(titulo, max_len=60) or "sem-titulo"
    sufixo_hash = hash_curto(metadados.get("url") or titulo)

    nome_base = f"{slug_source}_{slug_titulo}_{sufixo_hash}"
    caminho_txt = os.path.join(pasta, f"{nome_base}.txt")
    caminho_json = os.path.join(pasta, f"{nome_base}.json")

    with open(caminho_txt, "w", encoding="utf-8") as f:
        f.write(texto or "")
    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(metadados, f, ensure_ascii=False, indent=2)

    return caminho_txt, caminho_json


def processar_item(item: dict, config: dict, pasta_destino: str) -> Optional[dict]:
    titulo_listagem = item["titulo"]
    url = item["url"]
    print(f"\n  [item] {titulo_listagem[:100]}")

    html = baixar_pagina(url)
    if not html:
        print("    -> download falhou; pulando.")
        return None

    titulo = titulo_listagem
    if config["extrair_titulo"]:
        titulo = config["extrair_titulo"](html) or titulo_listagem

    texto = extrair_texto_generico(html)
    if not texto or len(texto) < MIN_CHARS_TEXTO:
        print(f"    -> texto muito curto ({len(texto)} chars); pulando.")
        return None

    data_publicacao = item.get("published_at")
    if config["extrair_data"]:
        data_publicacao = config["extrair_data"](html)

    if config.get("requer_data") and not data_publicacao:
        print("    -> sem data de publicação identificável; provável página de ferramenta/índice, pulando.")
        return None

    metadados = {
        "source_id": config["source_id"],
        "title": titulo,
        "description": config["source_descricao"],
        "url": url,
        "date": HOJE,
        "published_at": data_publicacao,
    }

    caminho_txt, caminho_json = salvar_artefatos(pasta_destino, config["source_id"], titulo, texto, metadados)
    print(f"    -> salvo em {caminho_txt}")
    return {"titulo": titulo, "url": url, "caminho_txt": caminho_txt, "caminho_json": caminho_json}

In [0]:
if NOME_FONTE == "todas":
    fontes_a_rodar = CONFIGS_FONTES
else:
    if NOME_FONTE not in CONFIGS_FONTES:
        raise ValueError(f"Fonte {NOME_FONTE!r} não configurada. Opções: {list(CONFIGS_FONTES)}")
    fontes_a_rodar = {NOME_FONTE: CONFIGS_FONTES[NOME_FONTE]}

resumo_geral = {}

for nome_fonte, config in fontes_a_rodar.items():
    print(f"\n{'='*70}\n=== Fonte: {nome_fonte!r} ===\n{'='*70}")

    caminho_manifesto = os.path.join(PASTA_MANIFESTOS, f"{config['source_id']}_processados.json")
    ja_processados = carregar_manifesto(caminho_manifesto)

    try:
        html_listagem = baixar_pagina(config["site_url"])
        if not html_listagem:
            resumo_geral[nome_fonte] = "ERRO: download da listagem falhou"
            atualizar_status_fonte(
                source_id=config["source_id"],
                sucesso=False,
                docs_capturados=0,
                erro="download da listagem falhou",
            )
            continue

        itens_pagina = config["listar"](html_listagem, config["site_url"])
        itens_novos = [i for i in itens_pagina if i["url"] not in ja_processados]
        print(f"{len(itens_pagina)} itens na página, {len(itens_novos)} novos.")

        salvos = 0
        for item in itens_novos:
            try:
                resultado = processar_item(item, config, PASTA_DESTINO)
                if resultado:
                    salvos += 1
                    ja_processados.add(item["url"])
            except Exception as e:
                print(f"[ERRO] item {item['titulo']!r} falhou: {e}")

        salvar_manifesto(caminho_manifesto, ja_processados)
        resumo_geral[nome_fonte] = f"{salvos} novos salvos"

        atualizar_status_fonte(
            source_id=config["source_id"],
            sucesso=True,
            docs_capturados=salvos,
        )

    except Exception as e:
        resumo_geral[nome_fonte] = f"ERRO: {e}"

        atualizar_status_fonte(
            source_id=config["source_id"],
            sucesso=False,
            docs_capturados=0,
            erro=str(e),
        )

print(f"\n\n{'='*70}\n=== RESUMO ===")
for nome_fonte, resultado in resumo_geral.items():
    print(f"  {nome_fonte}: {resultado}")
